# Workflow style

In [1]:
from dorna2 import pose as dorna_pose
import numpy as np
import time
from dorna_vision.limit import *
import json

def output(config, robot, timeout=-1): 
    cmd_list = []   
    for c in config:
        cmd_list.append({"cmd": "output", "out" + str(c[0]): c[1], "queue": 0})
        if len(c) > 2 and c[2] > 0:
            cmd_list.append({"cmd": "sleep", "time": c[2], "queue": 0})
    return robot.play_list(cmd_list, timeout=timeout)


def tray_to_tube_exist(robot, det, config):
    # imaging
    loc_img = list(config.tube_exist_to_well_plate["end"]["loc"])
    joint_img = loc_img[0] + loc_img[1]

    # safe
    robot.go(joint=joint_img)

    # detection
    retval = det.run()
    for r in retval:
        # pick location
        rvec = dorna_pose.align_abc(r["rvec"], align=[0, 0, 1], axis=[0, 0, -1], fix=[1, 0, 0])
        pick_pose = r["xyz"] + rvec        

        # adjustpick_pose
        pick_pose[2] -= 4

        # pick from the center of the tube
        pick_pose = dorna_pose.transform_pose([0, 20, 0, 0, 0, 0], from_frame=pick_pose, to_frame=[0,0,0,0,0,0])

        # update pick pose in config
        config.tray_to_tube_exist["pick"]["loc"][0] = list(pick_pose)
        return robot.pick_n_place( **config.tray_to_tube_exist)
    return None


def tube_exist_to_well_plate(index, robot, config):
    # place location
    config.tube_exist_to_well_plate["pick"]["loc"][0] = config.well_plate["place"][index]
    return robot.pick_n_place( **config.tube_exist_to_well_plate)


def tube_exist_to_tray(robot, config):
    return robot.pick_n_place( **config.tube_exist_to_tray)


def well_plate_to_decapper(index, robot, config):
    # two finger gripper open
    output(config.two_finger_gripper["open"], robot)

    # decapper open
    output(config.decapper["open"], robot)

    # pick location
    config.well_plate_to_decapper["pick"]["loc"][0] = list(config.well_plate["tube"][index])

    # go down more
    config.well_plate_to_decapper["pick"]["loc"][0][2] -= 4

    robot.pick_n_place( **config.well_plate_to_decapper)
        
    # open the gripper
    output(config.two_finger_gripper["open"], robot)

    # go up 5mm
    return robot.pick_n_place(**config.well_plate_second_part)


def release_cap(robot, config):
    return robot.pick_n_place( **config.release_cap)


def scan_tube(robot, config):
    return robot.pick_n_place( **config.scan_tube)


def tube_wo_cap_to_decapper(robot, config):
    return robot.pick_n_place( **config.tube_wo_cap_to_decapper)


def capping(robot, config):
    robot.pick_n_place( **config.capping)


def decapper_to_well_plate(index, robot, config):
    # pick location
    config.decapper_to_well_plate["pick"]["loc"][0] = list(config.well_plate["index"][index])
    # drop higher
    config.decapper_to_well_plate["pick"]["loc"][0][2] += 2
    return robot.pick_n_place( **config.decapper_to_well_plate)


def tool_changer_connect(tool, robot, config):
    # safe
    robot.go(joint=config.tool_changer["safe_joint"])

    # tool changer shaft in
    output(config.tool_changer_gripper["disconnect"], robot)

    # tool changer in idle mode
    output(config.tool_changer_gripper["idle"], robot)

    config.tool_changer_connect["pick"]["loc"] =  [config.tool_changer[tool]["connect"], config.tool_changer[tool]["aux"]]
    config.tool_changer_connect["pick"]["frame"] =  config.tool_changer[tool]["frame"]

    robot.pick_n_place( **config.tool_changer_connect)
    return robot.go(joint=config.tool_changer["safe_joint"])


def tool_changer_disconnect(tool, robot, config):
    # safe
    robot.go(joint=config.tool_changer["safe_joint"])

    disconnect_loc = list(config.tool_changer[tool]["connect"])
    disconnect_loc[2] -= 2.5
    config.tool_changer_disconnect["pick"]["loc"] =  [disconnect_loc, config.tool_changer[tool]["aux"]]
    config.tool_changer_disconnect["pick"]["frame"] =  config.tool_changer[tool]["frame"]
    
    robot.pick_n_place( **config.tool_changer_disconnect)
    return robot.go(joint=config.tool_changer["safe_joint"])


# Full workflow

In [3]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
from dorna_vision import Detection
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))

# camera
camera = Camera()
print("camera connected: ", camera.connect())

# detections
det_collection_tube_kp = Detection(robot=robot, camera=camera, **config.det_preset["collection_tube_kp"])
det_tube_exist = Detection(robot=robot, camera=camera, **config.det_preset["tube_exist"])

#####################
# step_1: tray picking
#####################
# connect suction gripper
tool_changer_connect("suction_gripper", robot, config)
all_indices = list(config.well_plate["index"].keys())

# picks
well_plate_index_list = []
while len(well_plate_index_list) < config.max_pick:
    # workflow
    result = tray_to_tube_exist(robot, det_collection_tube_kp, config)
    if result is not None:
        # tube exist detection
        retval = det_tube_exist.run()
        if retval: # tube exists
            # add new element
            well_plate_index_list.append(all_indices[len(well_plate_index_list)])
            tube_exist_to_well_plate(well_plate_index_list[-1], robot, config)
        else: # tube not exists
            tube_exist_to_tray(robot, config)
# disconnect suction gripper
tool_changer_disconnect("suction_gripper", robot, config)

#####################
# step_2: decapping
#####################
# connect two finger gripper
tool_changer_connect("two_finger_gripper", robot, config)

# safe joint
robot.go(joint=config.plate_holder["safe_joint"])

# decap all the indices
for index in well_plate_index_list:
    # well plate to decapper
    result = well_plate_to_decapper(index, robot, config)

    # release cap
    result = release_cap(robot, config)

    # scan tube
    result = scan_tube(robot, config)

    # tube without cap to decapper
    result = tube_wo_cap_to_decapper(robot, config)

    # capping
    result = capping(robot, config)

    # decapper to well plate
    result = decapper_to_well_plate(index, robot, config)

# disconnect two finger gripper
tool_changer_disconnect("two_finger_gripper", robot, config)
#####################
# workflow end
#####################

# close
robot.close()
camera.close()
det_collection_tube_kp.close()
det_tube_exist.close()


robot connected:  True
camera connected:  True


# Tray to well plate

In [2]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
from dorna_vision import Detection
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))

# camera
camera = Camera()
print("camera connected: ", camera.connect())

# detection
det_collection_tube_kp = Detection(robot=robot, camera=camera, **config.det_preset["collection_tube_kp"])
det_tube_exist = Detection(robot=robot, camera=camera, **config.det_preset["tube_exist"])

# keys
keys = list(config.well_plate["index"].keys())
max_pick = 8
counter = 0
while counter < max_pick:
    # workflow
    result = tray_to_tube_exist(robot, det_collection_tube_kp, config)
    if result is not None:
        # tube exist detection
        retval = det_tube_exist.run()
        if retval: # tube exists
            tube_exist_to_well_plate(keys[counter], robot, config)
            counter += 1
        else: # tube not exists
            tube_exist_to_tray(robot, config)

# close
robot.close()
camera.close()
det_collection_tube_kp.close()
det_tube_exist.close()

robot connected:  True
camera connected:  True


# Well plate to decapper

In [2]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = well_plate_to_decapper("a1", robot, config)

# close
robot.close()


robot connected:  True
{"cmd": "lmove", "rel": 0, "vel": 240.00000000000003, "accel": 1800.0000000000002, "jerk": 3000.0000000000005, "cont": 0, "j0": -68.94656166194761, "j1": 52.10030006462716, "j2": -118.48852942567066, "j3": -1.3270390367181335, "j4": -24.000146180408024, "j5": 112.5467215550262, "j6": 0.0, "j7": 0.0}
{"cmd": "lmove", "rel": 0, "vel": 240.00000000000003, "accel": 1800.0000000000002, "jerk": 3000.0000000000005, "cont": 0, "j0": -69.00369204359595, "j1": 40.25141628780376, "j2": -117.03266344191718, "j3": -2.2278826082289243, "j4": -13.646328738232455, "j5": 113.42671261635896, "j6": 0.0, "j7": 0.0}
{"cmd": "output", "out0": 0, "queue": 0}
{"cmd": "output", "out1": 0, "queue": 0}
{"cmd": "sleep", "time": 0.25, "queue": 0}
{"cmd": "sleep", "time": 0.5}
{"cmd": "lmove", "rel": 0, "vel": 240.00000000000003, "accel": 1800.0000000000002, "jerk": 3000.0000000000005, "cont": 0, "corner": 100, "j0": -68.95112719740831, "j1": 54.94203248261939, "j2": -118.54470943171629, "j3"

True

# Release cap

In [4]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = release_cap(robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    print(e)
    pass

# close
robot.close()

robot connected:  True
'float' object is not iterable


True

# Scan tube

In [5]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = scan_tube(robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    print(e)
    pass

# close
robot.close()


robot connected:  True
'float' object is not iterable


True

# Tube without cap to decapper

In [5]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = tube_wo_cap_to_decapper(robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    print(e)
    pass

# close
robot.close()

socket read error:  Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
robot connected:  True
'float' object is not iterable


True

# Capping

In [6]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = capping(robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    print(e)
    pass

# close
robot.close()

robot connected:  True
'NoneType' object is not iterable


True

# Decapper to well plate

In [7]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))


# workflow
result = decapper_to_well_plate("a5", robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    #print(e)
    pass

# close
robot.close()

robot connected:  True


True

# Tool changer connect disconnect

In [ ]:
# imports
import copy
from dorna2 import Dorna
from camera import Camera
import config
import json

# robot
robot = Dorna()
print("robot connected: ", robot.connect(config.robot["ip"]))

gripper_list = 2*["suction_gripper", "two_finger_gripper"]
for gripper in gripper_list:
    # workflow
    result = tool_changer_connect(gripper, robot, config)
    result = tool_changer_disconnect(gripper, robot, config)

try:
    for r in result:
        print(json.dumps(r))
except Exception as e:
    print(e)
    pass

# close
robot.close()

robot connected:  True
'float' object is not iterable


True